# Daily Occupancy Forecasting & Demand Dynamics Modeling
#### Notebook 4: Operational Scoring

This notebook loads previously trained models and applies them to the latest available data.

It does not retrain models. Its purpose is to produce:
- occupancy forecasts
- directional probabilities
- guardrail warnings
- recent reliability assessment
- calibrated operational actions

### Data Input and Update Instructions

This notebook is designed to score the latest available daily data using previously trained models.

#### A. Base dataset

The notebook expects the cleaned dataset generated in Notebook 1:

- File name: `daily_cleaned_dataset.csv`
- This file must contain the full historical data up to the most recent available date

Required columns:

- `Date`
- `Week`
- `Occupied_Units`
- `Occupancy`
- `Bookings`
- `Notices`
- `Avg_Booking_Time`
- `Avg_Notice_Time`
- `Has_Bookings`
- `Has_Notices`
- `units_available`


#### B. Optional: adding new data (e.g. 2020 onward)

To score more recent periods without retraining the model, you can provide an additional file:

- File name: `new_daily_data.csv`
- Same structure as `daily_cleaned_dataset.csv`
- It must include the same operational columns, especially `Date`, `Week`, `Occupied_Units`, `Occupancy`, `Bookings`, `Notices`, `Avg_Booking_Time`, `Avg_Notice_Time`, `Has_Bookings`, `Has_Notices`, and `units_available`

Important rules:

- Dates must be in a valid date format (YYYY-MM-DD recommended)
- Columns must match the base dataset exactly
- Only **new dates** (after the latest existing date) will be appended
- Duplicate dates will be automatically handled (latest version kept)

#### Important note

This notebook does **not retrain models**.

It assumes:
- models have already been trained (Notebook 02)
- new data follows the same structure and definitions

This notebook applies the final **calibrated operational logic** selected in Notebook 3. In addition to forecasts and directional probabilities, it also uses:

- a guardrail for structural instability
- an alignment score based on recent matured directional accuracy
- directional strength bands to distinguish stronger from weaker signals

This produces a more cautious and more realistic decision layer than using raw directional outputs alone.

#### Minimum data requirement for scoring

The model requires historical data to compute lag features (up to 28 days).

Therefore:
- new data must be appended to the existing dataset
- scoring cannot be performed on isolated rows without history

### 1. Imports

In [1]:
import pandas as pd
import numpy as np
import importlib
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import forecast_utils
importlib.reload(forecast_utils)

from forecast_utils import (
    add_features,
    load_trained_artifacts,
    apply_saved_seasonality,
    build_guardrail_from_saved_thresholds,
    build_operational_signal_multi_calibrated,
    add_direction_target
)

import build_operational_visual_board
importlib.reload(build_operational_visual_board)

from build_operational_visual_board import build_operational_visual_board
from IPython.display import display

### 2. Load cleaned dataset

In [2]:
# Load the cleaned daily dataset produced in Notebook 1.
daily = pd.read_csv("daily_cleaned_dataset.csv", parse_dates=["Date"])

print("Loaded dataset:", daily.shape)
print(daily.tail())

Loaded dataset: (1247, 12)
           Date  Occupied_Units      Week  Occupancy  Bookings  Notices  \
1242 2019-12-26            61.0  2019-W52   0.835616       0.0      0.0   
1243 2019-12-27            61.0  2019-W52   0.835616       0.0      0.0   
1244 2019-12-28            60.0  2019-W52   0.821918       0.0      0.0   
1245 2019-12-29            59.0  2019-W52   0.808219       0.0      0.0   
1246 2019-12-30            59.0  2020-W01   0.808219       0.0      0.0   

      Avg_Booking_Time  Avg_Notice_Time  Has_Bookings  Has_Notices  \
1242          8.333333             14.0             0            0   
1243          8.333333             14.0             0            0   
1244          8.333333             14.0             0            0   
1245          8.333333             14.0             0            0   
1246          8.333333             14.0             0            0   

      units_available  Missing_Occupancy_Flag  
1242             73.0                       0  
1243 

### 2b. Optional update with new daily rows

If a file containing newer daily observations is available, it can be appended to the historical cleaned dataset before scoring. This allows the notebook to reuse the trained models on more recent data without retraining.

In [3]:
new_data_path = Path("new_daily_data.csv")

if new_data_path.exists():
    new_data = pd.read_csv(new_data_path, parse_dates=["Date"])

    if not new_data.empty:
        print("New data found:", new_data.shape)

        # Keep only columns that exist in the base daily dataset
        common_cols = [col for col in daily.columns if col in new_data.columns]
        new_data = new_data[common_cols].copy()

        # Append and deduplicate by Date, keeping the newest version
        daily = pd.concat([daily, new_data], ignore_index=True)
        daily = (
            daily.sort_values("Date")
            .drop_duplicates(subset=["Date"], keep="last")
            .reset_index(drop=True)
        )

        print("Daily dataset after appending new rows:", daily.shape)
        print(daily.tail())
    else:
        print("new_daily_data.csv exists but is empty. Using original daily dataset.")
else:
    print("No new_daily_data.csv file found. Using original daily dataset.")

No new_daily_data.csv file found. Using original daily dataset.


### 3. Load saved models and metadata

In [4]:
# Load the trained models and all metadata saved from Notebook 2.
artifacts = load_trained_artifacts("models")

point_models = artifacts["point_models"]
direction_models = artifacts["direction_models"]
metadata = artifacts["metadata"]

features_t1 = metadata["features_t1"]
features_medium = metadata["features_medium"]
direction_feature_map = metadata["direction_feature_map"]
spike_threshold = metadata["spike_threshold"]
booking_guardrail_threshold = metadata["booking_guardrail_threshold"]
notice_guardrail_threshold = metadata["notice_guardrail_threshold"]
seasonal_by_day_train = metadata["seasonal_by_day_train"]
direction_threshold = metadata["direction_threshold"]

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


### 4. Build features

In [5]:
# Add engineered features to the cleaned daily dataset.
daily = add_features(daily)

# Apply the seasonal baseline learned on the training set in Notebook 2.
daily = apply_saved_seasonality(daily, seasonal_by_day_train)

# Add realized direction labels BEFORE creating the scoring dataframe.
# This is necessary for the alignment layer to work on historical rows.
daily = add_direction_target(
    daily,
    target_col="target_t1",
    new_col="Direction_t1",
    threshold=0.005
)

daily = add_direction_target(
    daily,
    target_col="target_t7",
    new_col="Direction_t7",
    threshold=0.005
)

# For scoring, we only need rows with the predictor columns required by the
# saved models. The direction columns are kept if they exist.
required_scoring_cols = list(set(features_t1 + features_medium))

daily_model = daily.dropna(subset=required_scoring_cols).copy()

print("Scoring dataset shape:", daily_model.shape)
print(daily_model[["Date", "Occupancy", "Direction_t1", "Direction_t7"]].tail())

Scoring dataset shape: (1233, 51)
           Date  Occupancy  Direction_t1  Direction_t7
1242 2019-12-26   0.835616             0             0
1243 2019-12-27   0.835616            -1             0
1244 2019-12-28   0.821918            -1             0
1245 2019-12-29   0.808219             0             0
1246 2019-12-30   0.808219             0             0


### 5. Select scoring window

In [6]:
# Select the most recent rows that have all required input features.
# These are the rows on which the saved models will generate forecasts.
score_df = daily_model.sort_values("Date").tail(90).copy()

print(score_df[["Date", "Occupancy", "Direction_t1", "Direction_t7"]].head())
print(score_df[["Date", "Occupancy", "Direction_t1", "Direction_t7"]].tail())

           Date  Occupancy  Direction_t1  Direction_t7
1157 2019-10-02   0.945205             1            -1
1158 2019-10-03   0.958904             0            -1
1159 2019-10-04   0.958904             1            -1
1160 2019-10-05   0.972603            -1            -1
1161 2019-10-06   0.958904            -1            -1
           Date  Occupancy  Direction_t1  Direction_t7
1242 2019-12-26   0.835616             0             0
1243 2019-12-27   0.835616            -1             0
1244 2019-12-28   0.821918            -1             0
1245 2019-12-29   0.808219             0             0
1246 2019-12-30   0.808219             0             0


### 6. Build guardrail from saved thresholds

In [7]:
# Apply the guardrail thresholds learned during training.
guardrail_score = build_guardrail_from_saved_thresholds(
    score_df,
    booking_threshold=booking_guardrail_threshold,
    notice_threshold=notice_guardrail_threshold
)

print("Guardrail counts:")
print(guardrail_score.value_counts())

Guardrail counts:
0    72
1    18
Name: count, dtype: int64


### 7. Generate point forecasts

In [8]:
# Generate multi-horizon occupancy forecasts using the saved point models.
pred_t7 = point_models["t7"].predict(score_df[features_medium])
pred_t14 = point_models["t14"].predict(score_df[features_medium])
pred_t21 = point_models["t21"].predict(score_df[features_medium])
pred_t28 = point_models["t28"].predict(score_df[features_medium])

# Optional:
# pred_t1 = point_models["t1"].predict(score_df[features_t1])

print("Generated point forecasts.")

Generated point forecasts.


### 8. Directional probabilities and predictions

### 8. Build calibrated operational output

This step applies the final operational scoring logic selected in Notebook 3.

The output combines:

- point forecasts
- directional probabilities
- directional strength bands
- guardrail-based instability
- recent reliability assessment
- calibrated operational actions

In [9]:
# This step applies the final operational scoring logic selected in Notebook 3.
final_output_multi = build_operational_signal_multi_calibrated(
    df=score_df,
    point_models=point_models,
    direction_models=direction_models,
    features_t1=features_t1,
    features_medium=features_medium,
    guardrail=guardrail_score,
    direction_threshold=direction_threshold,
    window=21,
    weak_thr=0.55,
    strong_thr=0.70
)

#final_output_multi.head(30)
#final_output_multi.tail(30)

#### 8.1 Diagnostic checks

In [10]:
# These checks help confirm that the calibrated layer is now working properly.
print(final_output_multi["Alignment_Band"].value_counts(dropna=False))
print(final_output_multi["Calibrated_Action"].value_counts(dropna=False))

Alignment_Band
HIGH       31
LOW        29
MEDIUM     19
UNKNOWN    11
Name: count, dtype: int64
Calibrated_Action
Reduce pricing / push demand        29
Monitor (low recent reliability)    26
Monitor (unstable)                  18
Increase pricing (strategic)        10
Prepare promotions                   6
Maintain current settings            1
Name: count, dtype: int64


### 9. Show latest operational recommendations

In [11]:
# Display the most recent actionable rows from the operational output.
latest_actions = final_output_multi.sort_values("Date").tail(50)

latest_actions[[
    "Date",
    "Occupancy",
    "Pred_t7",
    "Pred_t14",
    "Pred_t21",
    "Pred_t28",
    "Prob_Up_t21",
    "Prob_Down_t21",
    "Prob_Up_t28",
    "Prob_Down_t28",
    "Strength_t21",
    "Strength_t28",
    "Alignment_Band",
    "Guardrail",
    "Calibrated_Action"
]]

,Date,Occupancy,Pred_t7,Pred_t14,Pred_t21,Pred_t28,Prob_Up_t21,Prob_Down_t21,Prob_Up_t28,Prob_Down_t28,Strength_t21,Strength_t28,Alignment_Band,Guardrail,Calibrated_Action
40,2019-11-11,0.904110,0.914838,0.895380,0.859988,0.830208,0.433078,0.566922,0.507909,0.492091,MODERATE_DOWN,UNCERTAIN,MEDIUM,1,Monitor (unstable)
41,2019-11-12,0.904110,0.917715,0.894179,0.850028,0.816423,0.389596,0.610404,0.460254,0.539746,MODERATE_DOWN,UNCERTAIN,MEDIUM,1,Monitor (unstable)
42,2019-11-13,0.917808,0.929057,0.912614,0.871258,0.831976,0.437908,0.562092,0.456045,0.543955,MODERATE_DOWN,UNCERTAIN,LOW,0,Monitor (low recent reliability)
43,2019-11-14,0.904110,0.912340,0.896637,0.860956,0.826479,0.435599,0.564401,0.455008,0.544992,MODERATE_DOWN,UNCERTAIN,LOW,0,Monitor (low recent reliability)
44,2019-11-15,0.890411,0.904584,0.880519,0.849439,0.826661,0.440555,0.559445,0.466700,0.533300,MODERATE_DOWN,UNCERTAIN,LOW,1,Monitor (unstable)
45,2019-11-16,0.931507,0.929335,0.901103,0.864034,0.838922,0.368039,0.631961,0.389496,0.610504,MODERATE_DOWN,MODERATE_DOWN,LOW,1,Monitor (unstable)
46,2019-11-17,0.945205,0.934910,0.906405,0.876305,0.857514,0.381441,0.618559,0.401889,0.598111,MODERATE_DOWN,MODERATE_DOWN,LOW,0,Monitor (low recent reliability)
47,2019-11-18,0.945205,0.934844,0.903749,0.873177,0.853005,0.370892,0.629108,0.417675,0.582325,MODERATE_DOWN,MODERATE_DOWN,LOW,0,Monitor (low recent reliability)
48,2019-11-19,0.945205,0.938129,0.905573,0.867008,0.846261,0.338213,0.661787,0.373789,0.626211,MODERATE_DOWN,MODERATE_DOWN,LOW,0,Monitor (low recent reliability)
49,2019-11-20,0.931507,0.859528,0.806728,0.773424,0.778554,0.151662,0.848338,0.304081,0.695919,STRONG_DOWN,MODERATE_DOWN,LOW,1,Monitor (unstable)


### 10. Save scoring results

In [12]:
# Save the operational scoring output for external use.
final_output_multi.to_csv("scored_operational_output.csv", index=False)
print("Saved: scored_operational_output.csv")

Saved: scored_operational_output.csv


In a production setting, this process can be automated to run daily as new data becomes available, generating updated forecasts, reliability-aware warnings, and calibrated operational recommendations.

### 11. Operational board

In [13]:
display(build_operational_visual_board(final_output_multi, "2019-12-30"))

Control,Status,Operational Reading
Direction,,"Prob Up = 0.643, Prob Down = 0.357 → MODERATE_UP"
Reliability,,Recent alignment band = HIGH
Stability,,Stable (Guardrail = 0)
Action,,Increase pricing (strategic)
Control,Status,Operational Reading
Direction,,"Prob Up = 0.774, Prob Down = 0.226 → STRONG_UP"
Reliability,,Recent alignment band = HIGH
Stability,,Stable (Guardrail = 0)
Action,,Increase pricing (strategic)


### 

### 